# S AI LLM 1.0 · предобучение RU/EN

Запуск по одной ячейке в Colab с Tesla T4. Короткий пилот проверяет обновление внутренних весов, расход памяти и возобновление. Все результаты сохраняются в новой папке Google Drive. Исходные веса и прежние эксперименты остаются доступны по своим путям.

In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
root = Path("/content/drive/MyDrive/S_AI_Training")
disk = Path("/content/sai-1b")
disk.mkdir(exist_ok=True)
reports = root / "S_AI_LLM_1_0" / "pretraining_v3_20261108"
reports.mkdir(parents=True, exist_ok=True)
repo = disk / "S-AI-LLM"
branch = "codex/sai-llm-1b-pretraining"

def update_repo():
    if not repo.is_dir():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", branch,
                        "https://github.com/thepeshkov7/S-AI-LLM.git", str(repo)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", branch], check=True)
        subprocess.run(["git", "-C", str(repo), "checkout", "--detach", "FETCH_HEAD"], check=True)
    subprocess.run(["git", "-C", str(repo), "rev-parse", "HEAD"], check=True)

def run(module, *args):
    subprocess.run([sys.executable, "-u", "-m", "training_1b." + module, *map(str, args)],
                   cwd=repo, check=True)

def stage(source, name=None):
    target = disk / (name or source.name)
    if not target.is_file() or target.stat().st_size != source.stat().st_size:
        temporary = target.with_suffix(target.suffix + ".partial")
        shutil.copyfile(source, temporary)
        temporary.replace(target)
    return target

subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv"], check=True)
update_repo()
base = stage(root / "outputs/s_ai_llm_1_0_expansion/s_ai_llm_1_0_untrained_expanded.pt")
old = stage(root / "outputs/gpu_dialogue_v6/best.pt", "source-v6.pt")
v2 = stage(root / "S_AI_LLM_1_0/training_v2/best_output_head.pt")
trial = stage(root / "S_AI_LLM_1_0/output_head_trial_20_steps.pt")

In [ ]:
# 1. Проверка переноса и реальные ответы исходной модели и Training v2.
if not (reports / "checkpoint-audit.json").exists():
    run("checkpoint", "--base", base, "--previous", old, "--head", v2, "--head", trial,
        "--output", reports / "checkpoint-audit.json")
for label, head in (("expanded", None), ("training-v2", v2)):
    target = reports / (label + "-answers.json")
    if not target.exists():
        run("evaluate", "--checkpoint", base, "--output", target,
            *(("--head", head) if head else ()))

In [ ]:
# 2. Пилотный корпус из закреплённых срезов Wikipedia RU/EN.
update_repo()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes==0.48.2"], check=True)
corpus = disk / "wiki-pilot"
if not corpus.exists():
    run("corpus", "--checkpoint", base, "--output", corpus,
        "--cache", disk / "dataset-cache", "--wiki-shards", 2, "--limit-tokens", 5_000_000)
for name in ("manifest.json", "source-manifest.json"):
    shutil.copyfile(corpus / name, reports / ("wiki-" + name))

In [ ]:
# 3. Первый отрезок пилота. Обновляются все 20 Transformer-блоков.
output = reports / "t4-pilot"
configuration = repo / "training_1b/t4_pilot.json"
run("pretrain", "--profile", configuration, "--corpus", corpus, "--init", base,
    "--output", output, "--stop-after", 12)

In [ ]:
# 4. Возобновление из состояния, в том числе после перезапуска Colab.
state = stage(output / "training-state.pt", "resume-state.pt")
run("pretrain", "--profile", configuration, "--corpus", corpus, "--resume", state,
    "--output", output)

In [ ]:
# 5. Независимые ответы после пилота.
trained = stage(output / "training-state.pt", "trained-state.pt")
if not (reports / "t4-pilot-answers.json").exists():
    run("evaluate", "--checkpoint", trained, "--output", reports / "t4-pilot-answers.json")
print("Результаты сохранены в", reports)